This is a implementation of the Kuster-Toksoz model on the original probablistic inversion with mcmc.

In [ ]:
import numpy as np
import emcee
import matplotlib.pyplot as plt

In [ ]:
# ----- Fluid End-Member Properties -----
K_water = 2.2e9      # Bulk modulus of water in Pa
rho_water = 1000.0   # Density of water in kg/m³

K_gas = 0            # Bulk modulus for gas, Pa (Voigt mixing)
rho_gas = 0.020      # Density for gas in kg/m³

In [ ]:
# Polarization factors P and Q for Kuster-Toksoz model
# RPH Section 4.7, Table 4.7.1 (p.184)
def pq_from_alpha(alpha, k_m, mu_m, k_i, mu_i, eps=0.0):
    if alpha <= 0 or alpha > 1:
        return None, None, None

    # Auxiliary variables (RPH Table 4.7.1 notes, p.184):
    # beta = mu * (3K + mu) / (3K + 4*mu)
    beta_m = mu_m * (3.0*k_m + mu_m) / (3.0*k_m + 4.0*mu_m)
    # zeta = mu/6 * (9K + 8*mu) / (K + 2*mu)
    zeta_m = mu_m * (9.0*k_m + 8.0*mu_m) / (6.0*(k_m + 2.0*mu_m))

    if alpha <= 1e-2:
        # Penny crack expressions (RPH Table 4.7.1, "Penny cracks" row)
        shape = "penny"
        denom_P = k_i + (4.0/3.0)*mu_i + np.pi*alpha*beta_m + eps
        P = (k_m + (4.0/3.0)*mu_i) / denom_P
        term1 = 1.0
        term2 = 8.0*mu_m / (4.0*mu_i + np.pi*alpha*(mu_m + 2.0*beta_m) + eps)
        term3 = 2.0 * (k_i + (2.0/3.0)*(mu_i + mu_m)) / (k_i + (4.0/3.0)*mu_i + np.pi*alpha*beta_m + eps)
        Q = 0.2 * (term1 + term2 + term3)
    else:
        # Spherical inclusion expressions (RPH Table 4.7.1, "Spheres" row)
        shape = "sphere"
        P = (k_m + (4.0/3.0)*mu_m) / (k_i + (4.0/3.0)*mu_m + eps)
        Q = (mu_m + zeta_m) / (mu_i + zeta_m + eps)

    return float(P), float(Q), shape

In [ ]:
import numpy as np

# Kuster-Toksoz effective moduli
# RPH Section 4.7, p.183: main KT equations solved algebraically for K* and mu*
def kt_effective(
    k_m, mu_m,                 # matrix (host) bulk and shear moduli
    phi, S_w, alpha,           # porosity, water saturation, aspect ratio (oblate c/a)
    k_w, k_g,                  # water & gas bulk moduli
    mu_w=0.0, mu_g=0.0,        # water & gas shear moduli (fluids -> 0 in quasi-static)
    eps=1e-8                    # tiny stabilizer for near-singular denominators (optional)
):
    # zeta_m = mu/6 * (9K + 8mu) / (K + 2mu) — RPH p.183-184
    zeta_m = mu_m * (9.0 * k_m + 8.0 * mu_m) / (6.0 * (k_m + 2.0 * mu_m))

    # Compute P, Q for water and gas inclusions (RPH Table 4.7.1)
    P_w, Q_w, Sh_w = pq_from_alpha(alpha, k_m, mu_m, k_w, mu_w)
    P_g, Q_g, Sh_g = pq_from_alpha(alpha, k_m, mu_m, k_g, mu_g)

    # RHS of KT bulk equation: sum x_i*(K_i - K_m)*P^mi
    A = S_w * phi * (k_w - k_m) * P_w + (1.0 - S_w) * phi * (k_g - k_m) * P_g
    # RHS of KT shear equation: sum x_i*(mu_i - mu_m)*Q^mi
    B = S_w * phi * (mu_w - mu_m) * Q_w + (1.0 - S_w) * phi * (mu_g - mu_m) * Q_g

    # Solve algebraically for K* and mu* from KT equations (RPH p.183)
    # (K* - K_m) * (K_m + 4/3*mu_m) / (K* + 4/3*mu_m) = A
    a = k_m + (4.0 / 3.0) * mu_m
    k_e = (a * k_m + (4.0 / 3.0) * A * mu_m) / (a - A + eps)

    # (mu* - mu_m) * (mu_m + zeta_m) / (mu* + zeta_m) = B
    b = mu_m + zeta_m
    mu_e = (b * mu_m + B * zeta_m) / (b - B + eps)

    return k_e, mu_e

In [ ]:
def effective_density(rho_m, rho_i, phi):
    """
    Effective density ρ* = (1 - φ)*ρ_m + φ*ρ_i
    """
    return (1 - phi) * rho_m + phi * rho_i

In [ ]:
def seismic_velocities(K_eff, mu_eff, rho_eff):
    """
    Compute seismic velocities:
      Vp = sqrt((K_eff + 4/3*mu_eff)/ρ_eff)
      Vs = sqrt(mu_eff/ρ_eff)
    """
    Vp = np.sqrt((K_eff + (4.0/3.0)*mu_eff) / rho_eff)
    Vs = np.sqrt(mu_eff / rho_eff)
    return Vp, Vs

In [ ]:
def forward_model(x):
    """
    Forward model mapping unknown parameters to effective seismic properties.
    
    x = [S_w, phi, α, K_m, μ_m, ρ_m]
      S_w: water saturation (0 to 1)
      phi: crack porosity (volume fraction)
      α: crack aspect ratio
      K_m: matrix bulk modulus (Pa)
      μ_m: matrix shear modulus (Pa)
      ρ_m: matrix density (kg/m³)
      
    Fluid properties are computed as:
      K_i = S_w*K_water + (1-S_w)*K_gas
      ρ_i = S_w*rho_water + (1-S_w)*rho_gas
      
    Effective properties:
      K_eff = effective_bulk_modulus(K_m, μ_m, K_i, phi)
      μ_eff = effective_shear_modulus(μ_m, phi, α)
      ρ_eff = effective_density(ρ_m, ρ_i, phi)
      Vp, Vs = seismic_velocities(K_eff, μ_eff, ρ_eff)
      
    Returns: [Vp, Vs, ρ_eff]
    """
    x = np.asarray(x).ravel()  # Ensure x is a 1D array
    S_w, phi, logalpha, K_m, mu_m, rho_m = x
    alpha = 10**(logalpha)
    rho_i = S_w * rho_water + (1 - S_w) * rho_gas
    K_eff, mu_eff = kt_effective(K_m, mu_m, phi, S_w, alpha, K_water, K_gas, mu_w=0, mu_g=0)
    rho_eff = effective_density(rho_m, rho_i, phi)
    if (K_eff <= 0) or (mu_eff <= 0) or (rho_eff <= 0):
        return np.array([-np.inf, -np.inf, -np.inf])
    #print(K_eff, mu_eff, rho_eff)
    Vp, Vs = seismic_velocities(K_eff, mu_eff, rho_eff)
    return np.array([Vp, Vs, rho_eff])

In [ ]:
Vp_obs =4700.0       # m/s (5 km/s)
Vs_obs = 2700.0       # m/s (2.8 km/s)
# Assume observed effective density with matrix density ~3000 kg/m³ and water-filled pores:
rho_obs =  2589
sigma_Vp = 300.0       # m/s
sigma_Vs = 100.0       # m/s
sigma_rho = 157.0   

In [ ]:
def log_likelihood(x):
    model = forward_model(x)  # [Vp, Vs, ρ_eff]
    diff = model - np.array([Vp_obs, Vs_obs, rho_obs])
    chi2 = (diff[0]**2/sigma_Vp**2 +
            diff[1]**2/sigma_Vs**2 +
            diff[2]**2/sigma_rho**2)
    return -0.5 * chi2

In [ ]:
def log_prior(x):
    x = np.asarray(x).ravel()
    S_w, phi, logalpha, K_m, mu_m, rho_m = x
    # Uniform priors over plausible ranges:
    if (0.001 < S_w < 1.0 and 
        0.001 < phi < 0.5 and 
        -3.0 < logalpha < 0.0 and 
        756e8 < K_m < 80e9 and
        256e8 < mu_m < 40e9 and
        2680 < rho_m < 2900):
        return 0.0
    return -np.inf

In [ ]:
def log_probability(x):
    x = np.asarray(x).ravel()
    lp = log_prior(x)
    if not np.isfinite(lp):
        return -np.inf, np.full(3, -np.inf)
    ll = log_likelihood(x)
    blob = forward_model(x)  # Blob: [Vp, Vs, ρ_eff]
    return lp + ll, blob

In [ ]:
ndim = 6  # Unknown parameters: [S_w, phi, α, K_m, μ_m, ρ_m]
nwalkers = 3 * ndim

initial = np.zeros((nwalkers, ndim))
initial[:, 0] = np.random.uniform(0.001, 1.0, nwalkers)         # S_w
initial[:, 1] = np.random.uniform(0.001, 0.5, nwalkers)        # φ
initial[:, 2] = np.random.uniform(-3, 0, nwalkers)         # α
initial[:, 3] = np.random.uniform(756e8, 80e9, nwalkers)         # K_m
initial[:, 4] = np.random.uniform(256e8, 40e9, nwalkers)         # μ_m
initial[:, 5] = np.random.uniform(2680, 2900, nwalkers)         # ρ_m

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
nsteps = 50000
sampler.run_mcmc(initial, nsteps, progress=True)

# Retrieve flattened chain and blobs.
samples = sampler.get_chain(flat=True)
blobs = np.array(sampler.get_blobs(flat=True))  # Each blob: [Vp, Vs, ρ_eff]

# Identify best-fit sample.
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]
best_model = forward_model(best_params)

# The blobs array contains the effective seismic properties at each MCMC step.
# You can use them to construct PDFs of Vp, Vs, and effective density.
blobs = np.array(blobs)  # Ensure homogeneous shape; expected shape: (nsteps*nwalkers, 3)
Vp_pdf = blobs[:, 0]
Vs_pdf = blobs[:, 1]
rho_pdf = blobs[:, 2]

labels = [
    "Water Saturation",         #S_w
    "Crack Porosity",           # φ
    "Crack Aspect Ratio",       # α
    "Matrix Bulk Modulus (Pa)", # K_m
    "Matrix Shear Modulus (Pa)",# μ_m
    "Matrix Density (kg/m³)"    # ρ_m
]

fig, axes = plt.subplots(ndim, figsize=(10, 14), sharex=True)
for i in range(ndim):
    axes[i].plot(samples[:, i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()


In [ ]:
# ----- Get final, flat parameters from the sampler -----
chain = sampler.get_chain(flat=True)   # shape: (Nsamples, ndim)
# IMPORTANT: set the correct indices for your parameter order in log_probability
# Example: theta = [S_w, phi, u] where u = log10(alpha)
IDX_SW  = 0
IDX_PHI = 1
# If you truly sampled alpha directly (not log10), set IDX for alpha and ignore the transform below.

# Keep only finite rows (extra safety if you ever stored bad states)
finite_mask = np.all(np.isfinite(chain), axis=1)
chain_good  = chain[finite_mask]
if chain_good.size == 0:
    raise RuntimeError("No finite samples found in parameter chain.")

# Extract parameters explicitly
S_w  = chain_good[:, IDX_SW]
phi  = chain_good[:, IDX_PHI]

# If your third param is u = log10(alpha) and you want alpha elsewhere:
# u    = chain_good[:, 2]
# alpha = 10.0**u

# (Optional) trim burn-in/thinning here if you didn't already:
# burn = int(0.2 * chain_good.shape[0])
# S_w, phi = S_w[burn:], phi[burn:]

# ----- Plot and save exactly the first two: S_w and phi -----
import matplotlib.pyplot as plt

# Combined 2-row figure
fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=False)

# Water saturation
axes[0].hist(S_w, bins=50, density=True, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title("Water Saturation, $S_w$")
axes[0].set_ylabel("Density")

# Porosity
axes[1].hist(phi, bins=50, density=True, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].set_title("Porosity, $\\phi$")
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Density")

plt.tight_layout()
fig.savefig("water_saturation_and_porosity.png", dpi=200, bbox_inches='tight')
plt.close(fig)

In [ ]:
param_names = list(labels)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('Crack Porosity')
idx_water    = param_names.index('Water Saturation')

def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()

np.save("saturation_kt.npy", idx_water)
np.save("porosity_kt.npy", idx_porosity)
# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='porosity_kt_5.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='saturation_kt_5.png')

In [ ]:
def plot_1d_histograms(samples, labels=None, bins=100):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]  
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    for i in range(n_parameters):
        ax = axes[i] if n_parameters > 1 else axes  
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value") if n_parameters > 1 else axes.set_xlabel("Parameter Value")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_1d_histograms(samples, labels = labels, bins=50)

In [ ]:
bb1 = blobs.reshape(-1, 3)
mask = np.all(np.isfinite(bb1), axis=1)
bb_good = bb1[mask]
post_lab = ['vp', 'vs', 'rho_e']
plot_1d_histograms(bb_good, labels=post_lab, bins=50)

In [ ]:
K_water = 2.2e9      # Bulk modulus of water in Pa
rho_water = 1000.0   # Density of water in kg/m³

K_gas = 0            # Bulk modulus for gas, Pa (Voigt mixing)
rho_gas = 0.020      # Density for gas in kg/m³
best_K_i = best_params[0]   # Fluid bulk modulus, K_i (Pa)
best_rho_i = best_params[1] # Fluid density, rho_i (kg/m³)
S_w_from_K = (best_K_i - K_gas) / (K_water - K_gas)
S_w_from_rho = (best_rho_i - rho_gas) / (rho_water - rho_gas)

print("Water saturation computed from fluid bulk modulus (K_i): {:.3f}".format(S_w_from_K))
print("Water saturation computed from fluid density (rho_i): {:.3f}".format(S_w_from_rho))

# Optionally, you could average these two values if they are in close agreement:
S_w_avg = 0.5 * (S_w_from_K + S_w_from_rho)
print("Average water saturation: {:.3f}".format(S_w_avg))

In [ ]:
print(10*0.03*0.859)

In [ ]:
def plot_2d_color_plots(samples, labels):
    """
    Plot 2D color maps (histograms) for each unique pair of MCMC parameters.
    
    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      labels:  list of parameter labels (length ndim).
    """
    ndim = samples.shape[1]
    
    # Extract each parameter's chain data into a list.
    variable_data = [samples[:, i] for i in range(ndim)]
    
    # Create all unique pairs (i, j) with i < j.
    pairs = [(i, j) for i in range(ndim) for j in range(i+1, ndim)]
    npairs = len(pairs)
    
    # Determine a grid size (try a roughly square grid).
    ncols = int(np.ceil(np.sqrt(npairs)))
    nrows = int(np.ceil(npairs / ncols))
    
    fig, axs = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows))
    axs = np.ravel(axs)  # Flatten the axis array for easier indexing.
    
    for idx, (i, j) in enumerate(pairs):
        # Use the actual data arrays from the MCMC samples.
        x_data = variable_data[i]
        y_data = variable_data[j]
        
        # Compute 2D histogram with 100 bins for each axis.
        hist, x_edges, y_edges = np.histogram2d(x_data, y_data, bins=100)
        
        # Plot the 2D histogram as a color map.
        im = axs[idx].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                              origin="lower", aspect="auto", cmap="viridis")
        axs[idx].set_xlabel(labels[i])
        axs[idx].set_ylabel(labels[j])
        axs[idx].set_title(f"{labels[j]} vs {labels[i]}")
    
    # Turn off any unused subplots.
    for k in range(idx+1, len(axs)):
        axs[k].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs.tolist(), orientation="horizontal", fraction=0.02, pad=0.04)
    plt.show()

# Example usage (assuming 'samples' is your flattened MCMC chain of shape (N, ndim)):
labels = [
    "Fluid Bulk Modulus (Pa)",  # Parameter 0
    "Fluid Density (kg/m³)",    # Parameter 1
    "Crack Porosity",           # Parameter 2
    "Crack Aspect Ratio",       # Parameter 3
    "Matrix Bulk Modulus (Pa)", # Parameter 4
    "Matrix Shear Modulus (Pa)",# Parameter 5
    "Matrix Density (kg/m³)"    # Parameter 6
]

# Call the function (samples must be available)
plot_2d_color_plots(samples, labels)

In [ ]:
def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[0], best_params[1]))

In [ ]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

phi_mode = marginal_mode(samples[:,1], bins=100)
S_w_mode = marginal_mode(samples[:,0], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))

In [ ]:
# --- Distribution center (mean/median) based water estimate ---
phi_mean   = np.mean(samples[:,1])
S_w_mean   = np.mean(samples[:,0])
phi_median = np.median(samples[:,1])
S_w_median = np.median(samples[:,0])

print("Mean phi:", phi_mean)
print("Mean S_w:", S_w_mean)
print("Mean-based water estimate (m):", W_thickness(S_w_mean, phi_mean))

print("\nMedian phi:", phi_median)
print("Median S_w:", S_w_median)
print("Median-based water estimate (m):", W_thickness(S_w_median, phi_median))

In [ ]:
# --- Distribution-based water layer thickness ---
# Compute thickness for EVERY MCMC sample, then take statistics
thickness_samples = 8500 * samples[:,0] * samples[:,1]  # 8500 * S_w * phi

print('=== Distribution-based water layer thickness ===')
print('Mean thickness (m):', np.mean(thickness_samples))
print('Median thickness (m):', np.median(thickness_samples))
print('Mode thickness (m):', marginal_mode(thickness_samples, bins=100))
print('Std thickness (m):', np.std(thickness_samples))
print('95% CI (m):', np.percentile(thickness_samples, [2.5, 97.5]))
print('5th percentile (m):', np.percentile(thickness_samples, 5))
print('95th percentile (m):', np.percentile(thickness_samples, 95))

# Save samples array to disk for post-processing
np.save(os.path.join(output_dir, f'samples_{case_name}.npy'), samples)
np.save(os.path.join(output_dir, f'thickness_samples_{case_name}.npy'), thickness_samples)
print(f'Saved samples and thickness arrays to {output_dir}')